In [0]:
raw_table = dbutils.widgets.get("raw_table")
landing_table = dbutils.widgets.get("landing_table")

In [0]:
display(
spark.sql(f"""
INSERT INTO {raw_table}
SELECT
  'BEARS' AS source_system,
  CAST(office AS INT) AS office,
  clientno AS clientno,
  clientname AS clientname,
  payorname AS payorname,
  billto AS billto,
  payor_type_code AS payor_type_code,
  collector_name AS collector_name,
  invno AS invno,
  CAST(npi_number AS BIGINT) AS npi_number,
  try_to_date(first_visit_date, 'MM-dd-yy') AS first_visit_date,
  try_to_date(last_visit_date, 'MM-dd-yy') AS last_visit_date,
  try_to_date(last_payment_date, 'MM-dd-yy') AS last_payment_date,
  CAST(total_days_serviced AS INT) AS total_days_serviced,
  CAST(REPLACE(net_revenue, ',', '') AS DOUBLE) AS net_revenue,
  CAST(REPLACE(totaldue, ',', '') AS DOUBLE) AS account_balance,
  CAST(origbill AS DOUBLE) AS origbill,
  CAST(REPLACE(net_payments, ',', '') AS DOUBLE) AS total_payments,
  CAST(REPLACE(net_adjustments, ',', '') AS DOUBLE) AS total_adjustments,
  try_to_date(invdate, 'MM/dd/yy') AS invoice_date,
  SUM(COALESCE(CAST(REPLACE(current, ',', '') AS DOUBLE), 0))
    + SUM(COALESCE(CAST(REPLACE(four_seven_wks, ',', '') AS DOUBLE), 0))
    + SUM(COALESCE(CAST(REPLACE(eight_thirteen_wks, ',', '') AS DOUBLE), 0)) AS ar_0_90,
  SUM(COALESCE(CAST(REPLACE(fourteen_wks_to_reserve, ',', '') AS DOUBLE), 0)) AS ar_91_180,
  SUM(COALESCE(CAST(REPLACE(up_for_reserve, ',', '') AS DOUBLE), 0)) AS ar_181_270,
  SUM(COALESCE(CAST(REPLACE(prior_reserve, ',', '') AS DOUBLE), 0)) AS ar_271_plus,
  try_to_date(RIGHT(_file_name, 6), 'MMddyy') AS date_entered,
  _load_timestamp AS _load_timestamp,
  _file_name AS _file_name
FROM {landing_table} landing
WHERE NOT EXISTS (
    SELECT 1
    FROM {raw_table} raw
    WHERE raw._load_timestamp = landing._load_timestamp
    AND raw._file_name = landing._file_name 
)
GROUP BY
  office,clientno,clientname,invno,payorname,payor_type_code,collector_name,billto,npi_number,first_visit_date,last_visit_date,total_days_serviced,net_revenue,totaldue,origbill,total_payments,total_adjustments,invoice_date,last_payment_date,date_entered,_load_timestamp,_file_name 
""")
)